# Neural Network Inference: FPGA vs CPU
Transfer this notebook plus `design_1_wrapper.bit`, `design_1_wrapper.hwh`, and the four `.npy` weight files to the PYNQ board before running.

In [ ]:
from pynq import Overlay, allocate
import numpy as np
import time, sys

print("Loading overlay...", end=" ", flush=True)
t0 = time.time()
ol = Overlay("/home/xilinx/jupyter_notebooks/neural_network/design_1_wrapper.bit")
print(f"done  [{(time.time()-t0)*1000:.0f} ms]")
print(f"IPs found: {list(ol.ip_dict.keys())}")

In [ ]:
nn_ip = ol.nn_forward_0
print("IP register map:")
print(nn_ip.register_map)

In [ ]:
print("Loading weights for CPU reference inference...", end=" ", flush=True)
W1 = np.load("W1.npy").astype(np.float32)
b1 = np.load("b1.npy").astype(np.float32)
W2 = np.load("W2.npy").astype(np.float32)
b2 = np.load("b2.npy").astype(np.float32)
print(f"done  [W1={W1.shape}  W2={W2.shape}]")
print("Note: weights are embedded in the FPGA bitstream as BRAM — not transferred over AXI.")

In [ ]:
print("Allocating DDR buffers...", end=" ", flush=True)
x_buf = allocate(shape=(784,), dtype=np.float32)
y_buf = allocate(shape=(10,),  dtype=np.float32)
print("done")
print("Only x (3 KB) and y (40 B) need DDR buffers — weights live in BRAM.")

In [ ]:
# ── Register scan ─────────────────────────────────────────────────
print("Scanning writable AXI-Lite registers...")
writable = []
for off in range(0x00, 0x60, 4):
    try:
        nn_ip.write(off, 0x12345678)
        if nn_ip.read(off) == 0x12345678:
            writable.append(hex(off))
            nn_ip.write(off, 0)
    except Exception:
        pass
print("Writable offsets found:", ', '.join(writable))

# With BRAM weights, only 4 data regs expected: 0x10 0x14 0x1c 0x20
expected = {'0x10', '0x14', '0x1c', '0x20'}
found    = set(writable)
if expected.issubset(found) and len(found) <= 6:
    print("  OK — matches 2-argument BRAM bitstream")
elif len(found) > 6:
    print("  WARNING: more writable regs than expected — may be OLD bitstream with DDR weights!")
else:
    print("  WARNING: fewer regs than expected — check bitstream")

# ── Write buffer physical addresses ───────────────────────────────
print(f"\nx_buf physical address : 0x{x_buf.physical_address:08X}")
print(f"y_buf physical address : 0x{y_buf.physical_address:08X}")

LOWER = {'x': 0x10, 'y': 0x1C}
UPPER = {'x': 0x14, 'y': 0x20}
ADDRS = {'x': x_buf.physical_address, 'y': y_buf.physical_address}
for name in ['x', 'y']:
    nn_ip.write(LOWER[name], ADDRS[name] & 0xFFFFFFFF)
    nn_ip.write(UPPER[name], 0)

# ── Read back to verify ────────────────────────────────────────────
print("\nVerifying register writes:")
for name in ['x', 'y']:
    rb = nn_ip.read(LOWER[name])
    expected_val = ADDRS[name] & 0xFFFFFFFF
    status = "OK" if rb == expected_val else f"MISMATCH (got 0x{rb:08X}, expected 0x{expected_val:08X})"
    print(f"  {name} @ 0x{LOWER[name]:02X}: {status}")


In [ ]:
x_test = np.fromfile("x.bin",        dtype=np.float32)
label  = int(np.fromfile("y_expected.bin", dtype=np.float32).argmax())
print(f"Test sample loaded  |  True label: {label}")

In [ ]:
N_FPGA = 1000

print("=" * 42)
print(f"  FPGA INFERENCE  x{N_FPGA}  (weights in BRAM)")
print("=" * 42)

# Load input once — same sample every iteration
x_buf[:] = x_test
x_buf.flush()

print(f"  Running {N_FPGA} iterations...", end=" ", flush=True)
fpga_start = time.time()
for _ in range(N_FPGA):
    x_buf.flush()
    y_buf.flush()
    nn_ip.write(0x00, 1)          # AP_START
    while not (nn_ip.read(0x00) & 0x2):
        pass
    y_buf.invalidate()
fpga_end = time.time()

fpga_total_ms   = (fpga_end - fpga_start) * 1000
fpga_time_ms    = fpga_total_ms / N_FPGA
raw_logits      = np.array(y_buf)
fpga_pred       = int(np.argmax(raw_logits))
print("done")

print(f"\n  Prediction : {fpga_pred}  ({'correct' if fpga_pred == label else 'WRONG'})")
print(f"  Total time : {fpga_total_ms:.1f} ms  ({N_FPGA} runs)")
print(f"  Per sample : {fpga_time_ms:.4f} ms")
print(f"  Raw logits : {raw_logits.round(3)}")
print("=" * 42)


In [ ]:
N_CPU = 1000

print("=" * 42)
print(f"  CPU INFERENCE  x{N_CPU}")
print("=" * 42)

def forward_numpy(x, W1, b1, W2, b2):
    h = np.dot(W1, x) + b1
    h = np.maximum(h, 0)
    return np.dot(W2, h) + b2

print(f"  Running {N_CPU} iterations...", end=" ", flush=True)
cpu_start = time.time()
for _ in range(N_CPU):
    y_cpu = forward_numpy(x_test, W1, b1, W2, b2)
cpu_end = time.time()

cpu_total_ms = (cpu_end - cpu_start) * 1000
cpu_time_ms  = cpu_total_ms / N_CPU
cpu_pred     = int(np.argmax(y_cpu))

print("done")
print(f"\n  Prediction : {cpu_pred}  ({'correct' if cpu_pred == label else 'WRONG'})")
print(f"  Total time : {cpu_total_ms:.1f} ms  ({N_CPU} runs)")
print(f"  Per sample : {cpu_time_ms:.4f} ms")
print("=" * 42)


In [ ]:
max_err = float(np.max(np.abs(raw_logits - y_cpu)))
speedup = cpu_time_ms / fpga_time_ms

print()
print("=" * 42)
print("  RESULTS SUMMARY")
print("=" * 42)
print(f"  True label       : {label}")
print(f"  FPGA prediction  : {fpga_pred}  {'✓' if fpga_pred==label else '✗'}")
print(f"  CPU  prediction  : {cpu_pred}  {'✓' if cpu_pred==label else '✗'}")
print("-" * 42)
print(f"  FPGA time        : {fpga_time_ms:.4f} ms  (avg/{N_FPGA} runs)")
print(f"  CPU  time        : {cpu_time_ms:.4f} ms  (avg/{N_CPU} runs)")
print(f"  Speedup          : {speedup:.2f}x  ({'FPGA faster' if speedup>1 else 'CPU faster'})")
print("-" * 42)
print(f"  Max output error : {max_err:.6f}")
print(f"  Results match    : {'YES ✓' if max_err < 1e-3 else 'NO ✗ — check implementation'}")
print("=" * 42)


In [ ]:
x_buf.freebuffer()
y_buf.freebuffer()